# 06 · A forecast that only uses what was known at 9:30 am
ERCOT posts its day-ahead forecast at 9:30 am on the day before delivery. The model in notebook 02 used observed temperature and yesterday's load for every hour, so its comparison with ERCOT was not like for like. This notebook removes both advantages.

- **Load:** known only through hour ending 9 on the day before delivery. The rules are asserted in `src/op_forecast.py`.
- **Weather:** archived model forecasts (Open-Meteo Previous Runs), using the newest run already out at 9:30 am. GFS is the main source and JMA fills GFS's archive gap (2023-12-30 to 2024-01-19).
- **Target:** load divided by a baseline, the average of the same hour over recent fully known non-holiday days. Cross-validation below compares 14 and 28 days.
- **Holidays:** left out of the baseline and of the recent-load signals, so a holiday can't drag down the days after it.
- **Recent weather:** rolling 24 and 72 hour forecast temperature and the overnight low, because buildings hold heat and cold.
- **Choices:** inputs are compared with expanding-window cross-validation over 2022 and 2023, and settings are tuned on 2023. The 2024 to 2025 test set is only scored at the end.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import duckdb, numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
con = duckdb.connect("data/grid.duckdb", read_only=True)
con.execute("SET TimeZone = 'UTC'")
q = lambda sql: con.execute(sql).df()
import itertools, warnings
import plotly.graph_objects as go
import op_forecast as op
from viz import add_caption, save, SERIES, INK_2
warnings.filterwarnings("ignore")
base, holidays = op.load_frame(con)
base, city_cols, offset = op.add_forecast_weather(base, con, pd.read_parquet("data/raw/weather_forecast.parquet"),
                                                  pd.read_parquet("data/raw/weather.parquet"))
print(f"forecast minus observed offset used before the archive starts: {offset:+.2f} °C")
X = {n: op.add_load_features(base, holidays, n) for n in [14, 28]}
for x in X.values():
    op.assert_no_leak(x)
e = (base.fc_temp - base.avg_temp_c)[base.period >= "2021-04-01"]
yrs = base.period[base.period >= "2021-04-01"].dt.year
pd.DataFrame({"mean error (°C)": e.groupby(yrs).mean(), "mean absolute error (°C)": e.abs().groupby(yrs).mean()}).round(2)

forecast minus observed offset used before the archive starts: +0.61 °C


,mean error (°C),mean absolute error (°C)
period,,
2021,1.06,1.31
2022,1.02,1.31
2023,1.19,1.40
2024,0.87,1.33
2025,0.78,1.37


## Structure and tuning: expanding-window CV, one fold per quarter

In [2]:
def quarters(y):
    return [(f"{y}-01-01", f"{y}-04-01"), (f"{y}-04-01", f"{y}-07-01"), (f"{y}-07-01", f"{y}-10-01"), (f"{y}-10-01", f"{y + 1}-01-01")]
def cv(x, feats, params=None, folds=quarters(2023)):
    d = op.usable(x)
    errs = []
    for a, b in folds:
        tr, va = d[d.op_date < a], d[(d.op_date >= a) & (d.op_date < b)]
        m = op.fit(tr, feats, params)
        errs.append((va.demand_mwh - op.predict(m, va, feats)).abs() / va.demand_mwh)
    return round(100 * pd.concat(errs).mean(), 3)

SETS = {"core": op.CORE, "core + holiday-adjacent flags": op.CORE + op.ADJACENT,
        "core + flags + city temperatures": op.CORE + op.ADJACENT + city_cols,
        "core + flags + cities + recent weather": op.CORE + op.ADJACENT + city_cols + op.THERMAL}
structure = pd.DataFrame([{"baseline days": n, "inputs": k, "CV 2022": cv(X[n], f, folds=quarters(2022)), "CV 2023": cv(X[n], f)}
                          for n, (k, f) in itertools.product(X, SETS.items())])
structure["CV MAPE"] = structure[["CV 2022", "CV 2023"]].mean(axis=1).round(3)
structure.to_csv("data/clean/op_structure_cv.csv", index=False)
structure

,baseline days,inputs,CV 2022,CV 2023,CV MAPE
0,14,core,3.842,3.328,3.585
1,14,core + holiday-adjacent flags,3.823,3.305,3.564
2,14,core + flags + city temperatures,3.653,3.152,3.402
3,14,core + flags + cities + recent weather,3.335,3.174,3.254
4,28,core,3.965,3.300,3.632
5,28,core + holiday-adjacent flags,3.949,3.293,3.621
6,28,core + flags + city temperatures,3.775,3.174,3.474
7,28,core + flags + cities + recent weather,3.428,3.103,3.266


In [3]:
best = structure.loc[structure["CV MAPE"].idxmin()]
x, feats = X[int(best["baseline days"])], SETS[best["inputs"]]
print("chosen:", int(best["baseline days"]), "days,", best["inputs"])
grid = [dict(learning_rate=l, max_iter=i, max_leaf_nodes=n, min_samples_leaf=s)
        for l, i, n, s in itertools.product([0.03, 0.1], [200, 500], [15, 31], [20, 60])]
grid.append(dict(learning_rate=0.1, max_iter=800, max_leaf_nodes=31, min_samples_leaf=20))   # past the edge of the grid
tuning = pd.DataFrame([{**p, "CV MAPE": cv(x, feats, p)} for p in grid]).sort_values("CV MAPE")
tuning.to_csv("data/clean/op_tuning_cv.csv", index=False)
params = tuning.iloc[0].drop("CV MAPE").to_dict()
params = {k: (float(v) if k == "learning_rate" else int(v)) for k, v in params.items()}
print("default settings CV MAPE:", cv(x, feats)); print("chosen:", params)
tuning.head(5)

chosen: 14 days, core + flags + cities + recent weather


default settings CV MAPE: 3.174
chosen: {'learning_rate': 0.1, 'max_iter': 500, 'max_leaf_nodes': 31, 'min_samples_leaf': 60}


,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,CV MAPE
15,0.1,500,31,60,2.987
16,0.1,800,31,20,3.021
13,0.1,500,15,60,3.036
11,0.1,200,31,60,3.040
14,0.1,500,31,20,3.041


## Scoring the test set, once

In [4]:
d = op.usable(x)
train, test = d[d.year <= 2023], d[d.year >= 2024].copy()
model = op.fit(train, feats, params)
test["pred"] = op.predict(model, test, feats)
api = q("SELECT period, AVG(system_total_mwh) AS ercot_api_mwh FROM fact_ercot_forecast_model WHERE in_use GROUP BY 1")
api["period"] = pd.to_datetime(api.period, utc=True)
hind = pd.read_parquet("data/clean/test_predictions.parquet")[["period", "pred_best"]].rename(columns={"pred_best": "pred_hindsight"})
hind["period"] = pd.to_datetime(hind.period, utc=True)
lags = q("SELECT period, demand_lag_168 FROM model_hourly"); lags["period"] = pd.to_datetime(lags.period, utc=True)
t = test.merge(api, on="period").merge(hind, on="period").merge(lags, on="period").dropna(subset=["ercot_forecast_mwh"]).copy()
print(f"{len(t):,} test hours scored by every forecast")
y = t.demand_mwh
forecasts = {"Baseline: same hour last week": t.demand_lag_168,
             "Mine, hindsight version (notebook 02)": t.pred_hindsight,
             "Mine, 9:30 am information only": t.pred,
             "ERCOT 9:30 am posting (ERCOT API)": t.ercot_api_mwh,
             "ERCOT day-ahead, EIA-930 copy": t.ercot_forecast_mwh}
results = pd.DataFrame({k: {"RMSE (MWh)": round(op.rmse(y, v)), "MAPE (%)": round(op.mape(y, v), 2), "bias (MWh)": round((v - y).mean())}
                        for k, v in forecasts.items()}).T
results.to_csv("data/clean/op_results.csv")
results

17,489 test hours scored by every forecast


,RMSE (MWh),MAPE (%),bias (MWh)
Baseline: same hour last week,6482.0,8.53,-28.0
"Mine, hindsight version (notebook 02)",1971.0,2.52,50.0
"Mine, 9:30 am information only",2238.0,3.05,-304.0
ERCOT 9:30 am posting (ERCOT API),1810.0,2.38,410.0
"ERCOT day-ahead, EIA-930 copy",1757.0,2.31,418.0


In [5]:
t["ape"] = 100 * (t.pred - y).abs() / y
t["ape_ercot"] = 100 * (t.ercot_api_mwh - y).abs() / y
days = t.groupby("op_date")[["ape", "ape_ercot"]].mean()
print(f"days mine beats ERCOT's 9:30 am posting: {100 * (days.ape < days.ape_ercot).mean():.1f}% of {len(days)}")
print(f"hours: {100 * (t.ape < t.ape_ercot).mean():.1f}%")
by_year = t.groupby("year")[["ape", "ape_ercot"]].mean().round(2)
by_season = t.groupby("season")[["ape", "ape_ercot"]].mean().round(2).loc[["Winter", "Spring", "Summer", "Fall"]]
display(by_year); display(by_season)
summer_pm = t[t.month.between(6, 9) & t.he.between(16, 20)]
peak = t.groupby("op_date").apply(lambda g: pd.Series({
    "actual": g.demand_mwh.max(), "mine": g.pred.max(), "ercot": g.ercot_api_mwh.max(), "hour_ok_mine": g.pred.idxmax() == g.demand_mwh.idxmax(),
    "hour_ok_ercot": g.ercot_api_mwh.idxmax() == g.demand_mwh.idxmax()}))
peak_rows = {"All hours": (op.mape(y, t.pred), op.mape(y, t.ercot_api_mwh)),
             "Summer, 4 to 8 pm": (op.mape(summer_pm.demand_mwh, summer_pm.pred), op.mape(summer_pm.demand_mwh, summer_pm.ercot_api_mwh)),
             "Size of each day's peak": (op.mape(peak.actual.astype(float), peak.mine.astype(float)), op.mape(peak.actual.astype(float), peak.ercot.astype(float)))}
peaks = pd.DataFrame(peak_rows, index=["mine", "ERCOT"]).T.round(2)
peaks.to_csv("data/clean/op_peak_error.csv")
print(f"peak hour called exactly: mine {100 * peak.hour_ok_mine.astype(float).mean():.0f}%, ERCOT {100 * peak.hour_ok_ercot.astype(float).mean():.0f}%")
peaks

days mine beats ERCOT's 9:30 am posting: 29.5% of 729
hours: 40.4%


,ape,ape_ercot
year,,
2024,3.14,2.24
2025,2.95,2.53


,ape,ape_ercot
season,,
Winter,3.59,2.81
Spring,2.74,2.35
Summer,2.96,2.22
Fall,2.92,2.16


peak hour called exactly: mine 37%, ERCOT 59%


,mine,ERCOT
All hours,3.05,2.38
"Summer, 4 to 8 pm",3.05,2.39
Size of each day's peak,2.98,2.46


## Did the holiday fix work?
Same model and settings with holidays left in the baseline and recent-load signals, scored on holidays, the two days after them, and all other days.

In [6]:
raw = op.add_load_features(base, holidays, int(best["baseline days"]), skip_holidays=False)
ru = op.usable(raw)
rtr, rte = ru[ru.year <= 2023], ru[ru.year >= 2024].copy()
rte["pred_nofix"] = op.predict(op.fit(rtr, feats, params), rte, feats)
c = t.merge(rte[["period", "pred_nofix"]], on="period")
c["ape_nofix"] = 100 * (c.pred_nofix - c.demand_mwh).abs() / c.demand_mwh
hd = pd.to_datetime(sorted(holidays))
after = set(hd + pd.Timedelta(days=1)) | set(hd + pd.Timedelta(days=2))
c["group"] = np.where(c.holiday == 1, "Holiday", np.where(c.op_date.isin(after), "One or two days after", "Other days"))
hol = c.groupby("group")[["ape_nofix", "ape", "ape_ercot"]].mean().round(2)
hol["hours"] = c.groupby("group").size()
hol.to_csv("data/clean/op_holiday_fix.csv")
hol

,ape_nofix,ape,ape_ercot,hours
group,,,,
Holiday,4.03,4.05,2.91,528
One or two days after,3.39,3.47,2.92,1056
Other days,2.98,2.99,2.33,15905


## What drives the 9:30 am model
Grouped permutation importance on the test set: shuffle a group of inputs, measure the rise in MAPE, five shuffles each. The baseline stays fixed. Correlated inputs split credit, so a low score means the input adds little that the others don't carry, not that it is irrelevant.

In [7]:
groups = {c.replace("t_", "").replace("_", " ").title(): [c] for c in city_cols}
groups.update({"Statewide temperature": ["fc_temp", "cool", "heat", "temp_sq", "temp_vs_baseline"],
               "Hour of day": ["he"], "Day of week": ["dow", "weekend"], "Month": ["month"],
               "Holiday flags": ["holiday", "day_after_holiday", "day_before_holiday"],
               "Recent weather": op.THERMAL, "Morning load, day before": ["morning_ratio"], "Load two days before": ["d2_ratio"],
               "Baseline temperature": ["baseline_temp"]})
cities = set(c.replace("t_", "").replace("_", " ").title() for c in city_cols)
base_mape = op.mape(t.demand_mwh, op.predict(model, t, feats))
rng = np.random.default_rng(0)
rows = []
for name, cols in groups.items():
    rises = []
    for _ in range(5):
        s = t.copy(); idx = rng.permutation(len(s))
        for c_ in cols:
            s[c_] = s[c_].values[idx]
        rises.append(op.mape(t.demand_mwh, op.predict(model, s, feats)) - base_mape)
    rows.append({"input": name, "is_city": name in cities, "mape_increase_pts": np.mean(rises), "sd": np.std(rises)})
imp = pd.DataFrame(rows).sort_values("mape_increase_pts", ascending=False).round(3)
imp.to_csv("data/clean/op_feature_importance.csv", index=False)
imp

,input,is_city,mape_increase_pts,sd
8,Statewide temperature,False,2.266,0.021
13,Recent weather,False,1.902,0.013
16,Baseline temperature,False,1.201,0.012
14,"Morning load, day before",False,1.136,0.013
10,Day of week,False,0.789,0.007
2,Dallas,True,0.691,0.008
9,Hour of day,False,0.524,0.012
7,Tyler,True,0.206,0.005
3,Houston,True,0.148,0.001
15,Load two days before,False,0.137,0.005


In [8]:
srt = imp.sort_values("mape_increase_pts")
fig = go.Figure()
for is_city, label, color in [(True, "City temperature", SERIES[0]), (False, "Other input", SERIES[1])]:
    s = srt[srt.is_city == is_city]
    fig.add_bar(y=s.input, x=s.mape_increase_pts, orientation="h", name=label, marker_color=color,
                error_x=dict(type="data", array=s.sd, color=INK_2, thickness=1),
                hovertemplate="%{y}: +%{x:.2f} pts<extra></extra>")
fig.update_layout(title="What drives the 9:30 am forecast (test set 2024–2025)", barmode="overlay", bargap=0.3,
                  xaxis_title="Rise in MAPE when the input is shuffled (percentage points)", yaxis_title="",
                  yaxis=dict(categoryorder="array", categoryarray=list(srt.input), tickmode="array", tickvals=list(srt.input), tickfont=dict(size=12)), margin=dict(l=190, b=170))
add_caption(fig, f"{imp.iloc[0].input} matters most, then {imp.iloc[1].input.lower()}. Correlated inputs split credit. Bars: ±1 sd over 5 shuffles.")
save(fig, "feature_importance", height=720)

In [9]:
t["ape_base"] = 100 * (t.demand_lag_168 - y).abs() / y
s = t.groupby("season")[["ape_base", "ape", "ape_ercot"]].mean().loc[["Winter", "Spring", "Summer", "Fall"]]
fig = go.Figure()
for i, (col, name) in enumerate([("ape_base", "Baseline (last week)"), ("ape", "Mine (9:30 am information)"), ("ape_ercot", "ERCOT 9:30 am posting")]):
    fig.add_bar(x=s.index, y=s[col], name=name, marker_color=SERIES[i], hovertemplate=name + " · %{x}: %{y:.2f}%<extra></extra>")
fig.update_layout(title="Forecast error by season, test set 2024–2025", barmode="group", bargap=0.35, bargroupgap=0.08,
                  yaxis_title="Mean absolute % error", xaxis_title="Season")
add_caption(fig, "All three use only information available at 9:30 am the day before. Winter is the hardest season for each.")
save(fig, "model_error_by_season")
s.round(2)

,ape_base,ape,ape_ercot
season,,,
Winter,12.14,3.59,2.81
Spring,7.49,2.74,2.35
Summer,6.70,2.96,2.22
Fall,7.87,2.92,2.16


## The 10 biggest misses

In [10]:
names = pd.read_parquet("data/clean/us_holidays.parquet")
names = dict(zip(pd.to_datetime(names.holiday_date), names.holiday_name))
t["err"] = t.pred - t.demand_mwh
miss = t.groupby("op_date").agg(mine=("ape", "mean"), ercot=("ape_ercot", "mean"), mean_error_gw=("err", lambda e: e.mean() / 1000),
                                tmin=("avg_temp_c", "min"), tmax=("avg_temp_c", "max"), fc_error=("fc_temp", "mean"), obs=("avg_temp_c", "mean"))
miss["temp_forecast_miss_c"] = miss.fc_error - miss.obs
worst = miss.sort_values("mine", ascending=False).head(10).drop(columns=["fc_error", "obs"]).round(1)
worst["holiday"] = [names.get(d, "") for d in worst.index]
worst.to_csv("data/clean/op_biggest_misses.csv")
t[["period", "op_date", "he", "season", "demand_mwh", "pred", "ercot_api_mwh", "ape", "ape_ercot"]].to_parquet("data/clean/op_test_predictions.parquet", index=False)
worst

,mine,ercot,mean_error_gw,tmin,tmax,temp_forecast_miss_c,holiday
op_date,,,,,,,
2024-07-08,19.3,16.3,9.6,24.5,28.9,-0.4,
2024-01-15,14.1,4.8,-10.2,-7.1,0.8,6.8,Martin Luther King Jr. Day
2024-05-17,10.2,10.2,4.8,16.8,26.4,2.1,
2024-01-14,9.8,6.0,-5.7,-5.0,2.7,7.0,
2024-05-28,9.7,9.0,3.6,22.7,29.5,2.9,
2024-07-09,9.1,9.7,4.8,22.4,32.8,0.4,
2024-01-29,9.1,1.2,4.0,4.5,19.0,1.1,
2024-05-16,8.2,5.6,1.3,19.2,25.3,1.3,
2024-11-01,8.0,2.9,4.1,18.6,26.2,1.6,
